# **Visualisations des résultats**

## **Imports**

In [1]:
import os
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from pymongo import MongoClient

In [2]:

load_dotenv()
db = MongoClient(os.getenv("MONGO_URI"))["cinema"]

In [3]:
genres = pd.DataFrame(db["stats_genres"].find())
annees = pd.DataFrame(db["stats_annees"].find())
realisateurs = pd.DataFrame(db["top_realisateurs"].find())


In [4]:
genres.head()

,_id,nb_films,note_moyenne,budget_moyen,recettes_moyennes
0,Drame,2552,6.868708,2.818437e+07,6.318945e+07
1,Comédie,2055,6.460270,4.139318e+07,1.135585e+08
2,Thriller,1701,6.430909,3.503227e+07,8.096490e+07
3,Action,1526,6.522823,7.035215e+07,1.900930e+08
4,Aventure,1068,6.634307,9.171088e+07,2.816162e+08


In [5]:
genres = genres.rename(columns={"_id": "genre"})
genres["recettes_M"] = genres["recettes_moyennes"] / 1e6   # en millions de $
genres["budget_M"] = genres["budget_moyen"] / 1e6
genres.head()

,genre,nb_films,note_moyenne,budget_moyen,recettes_moyennes,recettes_M,budget_M
0,Drame,2552,6.868708,2.818437e+07,6.318945e+07,63.189449,28.184367
1,Comédie,2055,6.460270,4.139318e+07,1.135585e+08,113.558459,41.393181
2,Thriller,1701,6.430909,3.503227e+07,8.096490e+07,80.964900,35.032274
3,Action,1526,6.522823,7.035215e+07,1.900930e+08,190.092954,70.352146
4,Aventure,1068,6.634307,9.171088e+07,2.816162e+08,281.616218,91.710879


### *Nombre de films par genres*

In [17]:
fig = px.bar(
    genres.sort_values("nb_films"),         
    x="nb_films", y="genre", orientation="h", 
    title="Nombre de films par genre (2000-2025, ≥ 500 votes)",
    labels={"nb_films": "Nombre de films", "genre": ""},
    text="nb_films",color_discrete_sequence=["#6FABFC"]                    
)
fig.update_layout(height=600)
fig.update_traces(textposition="outside")
fig.show()

### *Recettes moyennes par genre*

In [16]:
fig = px.bar(
    genres.sort_values("recettes_M"),
    x="recettes_M", y="genre", orientation="h",
    title="Recettes moyennes par genre (en millions de $)",
    labels={"recettes_M": "Recettes moyennes (M$)", "genre": ""},
    text=genres.sort_values("recettes_M")["recettes_M"].round(0),color_discrete_sequence=["#6FABFC"]
)
fig.update_layout(height=600)
fig.update_traces(textposition="outside")
fig.show()

Ce que montrent les graphiques

1. Quantité et rentabilité ne vont pas ensemble.
Le Drame est le genre le plus produit (2 552 films), mais il est 16e sur 19 en recettes moyennes (63 M$). À l'inverse, l'Aventure n'est que 5e en nombre (1 068 films), mais 1re au box-office (282 M$), soit 4,5 fois plus que le Drame.

2. Le top 5 des recettes est celui des « blockbusters familiaux ».
Aventure, Animation, Familial, Science-Fiction et Fantastique dépassent tous 200 M$ en moyenne. Ce sont les genres des grandes franchises (Marvel, Pixar, Star Wars…), conçus pour un public large et un marché mondial.

3. Les genres « de niche » rapportent peu, mais pour des raisons différentes.

L'Horreur (58 M$) a des recettes modestes, mais ce sont souvent des films à petit budget. Il faudra regarder la rentabilité, pas seulement les recettes.
Le Téléfilm (6 M$) et le Documentaire (33 M$) sont des formats peu ou pas diffusés en salles, d'où des recettes faibles, voire inconnues.

4. Cas intéressant : la Musique (104 M$, 8e). Avec seulement 148 films, elle dépasse des genres bien plus produits comme le Thriller ou le Crime. Quelques gros succès suffisent à tirer la moyenne vers le haut.

Les limites à mentionner
Une moyenne est sensible aux extrêmes. Quelques films à plus d'un milliard de dollars font monter fortement la moyenne d'un genre. La médiane donnerait une image plus représentative du film « typique ».
Les recettes inconnues sont exclues. Environ 20 % des films n'ont pas de recettes renseignées : ces moyennes ne portent que sur les films dont on connaît le box-office.
Un film compte dans plusieurs genres. Un film d'Action-Aventure pèse dans les deux barres.
Recettes ne veut pas dire rentabilité. Le graphique ne tient pas compte du budget.

### *Rentabilité médiane des films par année*

In [21]:
annees = annees.rename(columns={"_id": "annee"}).sort_values("annee")

fig = px.line(
    annees,
    x="annee", y="rentabilite_mediane",
    markers=True,                                  # un point par année
    title="Rentabilité médiane des films par année (recettes ÷ budget)",
    labels={"annee": "Année", "rentabilite_mediane": "Rentabilité médiane (×)"},
)

# Ligne du seuil de rentabilité (le film rapporte exactement son budget)
fig.add_hline(y=1, line_dash="dash", line_color="red",
              annotation_text="Seuil de rentabilité", annotation_position="bottom right")

# Zone grisée sur la période Covid
fig.add_vrect(x0=2019.5, x1=2021.5, fillcolor="#6FABFC", opacity=0.2, line_width=0,
              annotation_text="Covid-19", annotation_position="top left")

fig.update_yaxes(rangemode="tozero")               # l'axe commence à 0 → échelle honnête
fig.show()

Le **choc du Covid (2020)**. La courbe plonge sous le seuil de rentabilité (× 0,81) : plus de la moitié des films de l'année ont perdu de l'argent, avec les cinémas fermés une grande partie de l'année.

Une reprise lente mais complète (2021-2025). × 1,09, puis 1,56, 1,80, 1,90 et 2,31 en 2025. Il a fallu environ cinq ans pour retrouver le niveau d'avant-crise.

In [13]:
realisateurs = realisateurs.sort_values("note_moyenne")

fig = px.scatter(
    realisateurs,
    x="note_moyenne", y="nom",
    size="nb_films",
    color="nb_films",
    color_continuous_scale="Blues",       # ← le dégradé de couleurs
    title="Top 20 des réalisateurs par note moyenne (au moins 5 films)",
    labels={"note_moyenne": "Note moyenne", "nom": "", "nb_films": "Nombre de films"},
    hover_data={"recettes_totales": ":,.0f"},
)
fig.update_layout(height=650)
fig.show()